# Key Parameters to Adjust for Better Results

If you get suboptimal results with the VDSR implementation, here are the key parameters you can modify to improve performance:

## 1. Model Architecture Parameters

- **NUM_LAYERS** (default: 20): Increasing the depth can capture more complex features but requires more VRAM and training time. Try values from **10-30**.
- **NUM_FILTERS** (default: 64): More filters can capture more features but increase memory usage. Try **32, 64, 96, or 128**.

## 2. Data Processing Parameters

- **IMAGE_SIZE** (default: 224): Larger sizes can preserve more detail but require more memory. Try **128, 256, or 320**.
- **DOWNSCALE_FACTOR** (default: 4): Controls how low-quality the input images are. Try **2** (less degradation) or **8** (more degradation) depending on your goal.

## 3. Training Parameters

- **LEARNING_RATE** (default: 0.001): If training is unstable or loss plateaus early, try **0.0001 or 0.0005**.
- **BATCH_SIZE** (default: 32): Smaller batch sizes can help with limited memory or when increasing model size. Try **8, 16, or 64**.
- **EPOCHS** (default: 50): Increase to **100-200** if the model is still improving at the end of training.
- **WEIGHT_DECAY** (default: 1e-4): Controls L2 regularization. Try **1e-5** or **1e-3** if overfitting or underfitting.
- **GRAD_CLIP_VALUE** (default: 0.4): Helps prevent exploding gradients. Try **0.1-1.0** if training is unstable.

## 4. Learning Rate Scheduler Parameters

- **LR_DECAY_STEP** (default: 10): How many epochs before reducing the learning rate. Try **5-20**.
- **LR_DECAY_GAMMA** (default: 0.5): Factor to reduce learning rate by. Try **0.1-0.8**.

## 5. Optimizer Options

The current implementation uses **Adam**, but you could try:

```python
# SGD with momentum
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

# RMSprop
optimizer = optim.RMSprop(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
```

## 6. Loss Function Alternatives
The current implementation uses MSE loss, but you could experiment with:

```python
# L1 Loss (Mean Absolute Error)
criterion = nn.L1Loss()

# Combination of L1 and MSE
def combined_loss(pred, target):
    mse_loss = nn.MSELoss()(pred, target)
    l1_loss = nn.L1Loss()(pred, target)
    return 0.7 * mse_loss + 0.3 * l1_loss
```

## 7. Data Augmentation
Add data augmentation to increase the effective dataset size:

```python
# Add to the CXRDataset class
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
])
```

## Debugging Strategies
If your results are poor:

- **Check intermediate outputs**: Add code to save low-res and super-res images at different epochs.
- **Monitor loss curves**: If validation loss increases while training loss decreases, you're overfitting.
- **Try with a small subset**: Test on 100 images first to confirm the pipeline works.
- **Examine poor-performing images**: Look for patterns in images with low PSNR/SSIM.
- **Increase degradation gradually**: Start with a small downscale factor (2) and gradually increase it.

# Implementation

## 1. IMPORTS AND CONFIGURATION

In [1]:

import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from sklearn.model_selection import train_test_split
import glob
import cv2
from tqdm import tqdm
import pandas as pd

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Parameters (These can be adjusted to improve results)
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4  # L2 regularization
MOMENTUM = 0.9  # For SGD optimizer
LR_DECAY_STEP = 10
LR_DECAY_GAMMA = 0.5
DATASET_PATH = "chest-x-ray-dataset/chest_xray"
DOWNSCALE_FACTOR = 4  # Factor for creating low-resolution images
NUM_LAYERS = 20  # VDSR depth
NUM_FILTERS = 64  # Number of filters in each layer
GRAD_CLIP_VALUE = 0.4  # For gradient clipping
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. MODEL ARCHITECTURE

In [2]:
class VDSR(nn.Module):
    def __init__(self, num_layers=NUM_LAYERS, num_filters=NUM_FILTERS):
        super(VDSR, self).__init__()
        
        # First layer
        layers = [
            nn.Conv2d(in_channels=1, out_channels=num_filters, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        ]
        
        # Middle layers
        for _ in range(num_layers - 2):
            layers.append(nn.Conv2d(in_channels=num_filters, out_channels=num_filters, kernel_size=3, padding=1))
            layers.append(nn.ReLU(inplace=True))
        
        # Last layer
        layers.append(nn.Conv2d(in_channels=num_filters, out_channels=1, kernel_size=3, padding=1))
        
        self.residual = nn.Sequential(*layers)
    
    def forward(self, x):
        # Residual learning: output = input + residual
        return x + self.residual(x)

## 3. DATASET CLASS

In [3]:
class CXRDataset(Dataset):
    def __init__(self, lr_images, hr_images, transform=None):
        self.lr_images = lr_images
        self.hr_images = hr_images
        self.transform = transform
    
    def __len__(self):
        return len(self.lr_images)
    
    def __getitem__(self, idx):
        lr_image = self.lr_images[idx]
        hr_image = self.hr_images[idx]
        
        # Convert to PyTorch tensors
        lr_tensor = torch.from_numpy(lr_image).float()
        hr_tensor = torch.from_numpy(hr_image).float()
        
        # PyTorch uses [C, H, W] format
        lr_tensor = lr_tensor.permute(2, 0, 1)
        hr_tensor = hr_tensor.permute(2, 0, 1)
        
        if self.transform:
            lr_tensor = self.transform(lr_tensor)
            hr_tensor = self.transform(hr_tensor)
        
        return lr_tensor, hr_tensor

## 4. DATA PREPARATION

In [4]:
def prepare_data(dataset_path, limit_per_class=500, downscale_factor=DOWNSCALE_FACTOR):
    """
    Load and prepare the training and validation data
    Returns: train_loader, val_loader
    """
    hr_images = []
    categories = ['NORMAL', 'PNEUMONIA']
    
    # Collect all images from training directory
    for category in categories:
        image_paths = glob.glob(os.path.join(dataset_path, 'train', category, '*.jpeg'))
        image_paths += glob.glob(os.path.join(dataset_path, 'train', category, '*.jpg'))
        
        print(f"Found {len(image_paths)} {category} images")
        
        for img_path in tqdm(image_paths[:limit_per_class]):  # Limit images per class for faster processing
            try:
                # Load image in grayscale
                img = cv2.imread(img_path, 0)  # 0 for grayscale
                
                if img is not None:
                    # Resize image to target size
                    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
                    
                    # Normalize to [0, 1]
                    img = img.astype(np.float32) / 255.0
                    
                    hr_images.append(img)
            except Exception as e:
                print(f"Error processing {img_path}: {e}")
    
    hr_images = np.array(hr_images)
    print(f"Total HR images: {hr_images.shape}")
    
    # Create low-resolution images by downsampling and then upsampling
    lr_images = []
    for img in tqdm(hr_images):
        # Downsample
        low_res = cv2.resize(img, (IMAGE_SIZE // downscale_factor, IMAGE_SIZE // downscale_factor))
        # Upsample back to original size (this creates the low-quality image)
        low_res = cv2.resize(low_res, (IMAGE_SIZE, IMAGE_SIZE))
        lr_images.append(low_res)
    
    lr_images = np.array(lr_images)
    
    # Reshape for the model (add channel dimension)
    hr_images = hr_images.reshape(-1, IMAGE_SIZE, IMAGE_SIZE, 1)
    lr_images = lr_images.reshape(-1, IMAGE_SIZE, IMAGE_SIZE, 1)
    
    # Split into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(lr_images, hr_images, test_size=0.2, random_state=42)
    
    # Create PyTorch datasets
    train_dataset = CXRDataset(X_train, y_train)
    val_dataset = CXRDataset(X_val, y_val)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    
    return train_loader, val_loader

## 5. TEST DATA LOADING

In [5]:
def load_test_data(dataset_path, limit_per_class=50, downscale_factor=DOWNSCALE_FACTOR):
    """
    Load and prepare test data for evaluation
    Returns: test_loader, test_hr_images, test_lr_images
    """
    test_hr_images = []
    test_lr_images = []
    categories = ['NORMAL', 'PNEUMONIA']
    
    for category in categories:
        image_paths = glob.glob(os.path.join(dataset_path, 'test', category, '*.jpeg'))
        image_paths += glob.glob(os.path.join(dataset_path, 'test', category, '*.jpg'))
        
        print(f"Found {len(image_paths)} {category} test images")
        
        for img_path in tqdm(image_paths[:limit_per_class]):  # Limit images per class
            try:
                # Load image in grayscale
                img = cv2.imread(img_path, 0)
                
                if img is not None:
                    # Resize image to target size
                    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
                    
                    # Normalize to [0, 1]
                    img = img.astype(np.float32) / 255.0
                    
                    # Create low resolution version
                    low_res = cv2.resize(img, (IMAGE_SIZE // downscale_factor, IMAGE_SIZE // downscale_factor))
                    low_res = cv2.resize(low_res, (IMAGE_SIZE, IMAGE_SIZE))
                    
                    test_hr_images.append(img)
                    test_lr_images.append(low_res)
            except Exception as e:
                print(f"Error processing test image {img_path}: {e}")
    
    # Reshape for the model
    test_hr_images = np.array(test_hr_images).reshape(-1, IMAGE_SIZE, IMAGE_SIZE, 1)
    test_lr_images = np.array(test_lr_images).reshape(-1, IMAGE_SIZE, IMAGE_SIZE, 1)
    
    # Create PyTorch dataset
    test_dataset = CXRDataset(test_lr_images, test_hr_images)
    
    # Create data loader
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    
    return test_loader, test_hr_images, test_lr_images

## 6. TRAINING FUNCTION

In [6]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=EPOCHS):
    """
    Train the VDSR model
    Returns: training history (losses)
    """
    model.to(DEVICE)
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        
        for lr_imgs, hr_imgs in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            lr_imgs = lr_imgs.to(DEVICE)
            hr_imgs = hr_imgs.to(DEVICE)
            
            # Zero the parameter gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(lr_imgs)
            loss = criterion(outputs, hr_imgs)
            
            # Backward pass and optimize
            loss.backward()
            
            # Gradient clipping to prevent exploding gradients
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_VALUE)
            
            optimizer.step()
            
            running_loss += loss.item() * lr_imgs.size(0)
        
        epoch_train_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for lr_imgs, hr_imgs in val_loader:
                lr_imgs = lr_imgs.to(DEVICE)
                hr_imgs = hr_imgs.to(DEVICE)
                
                outputs = model(lr_imgs)
                loss = criterion(outputs, hr_imgs)
                
                val_loss += loss.item() * lr_imgs.size(0)
        
        epoch_val_loss = val_loss / len(val_loader.dataset)
        val_losses.append(epoch_val_loss)
        
        # Update learning rate
        scheduler.step()
        
        # Print progress
        print(f"Epoch {epoch+1}/{num_epochs} => "
              f"Train Loss: {epoch_train_loss:.6f}, "
              f"Val Loss: {epoch_val_loss:.6f}, "
              f"LR: {optimizer.param_groups[0]['lr']:.8f}")
        
        # Save best model
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': best_val_loss,
            }, 'vdsr_model_best.pth')
            print(f"Saved new best model with val_loss: {best_val_loss:.6f}")
    
    return train_losses, val_losses

## 7. EVALUATION FUNCTION

In [7]:
def evaluate_model(model, test_loader, test_hr_images, test_lr_images):
    """
    Evaluate the model using test data
    Returns: super-resolution images, individual PSNR/SSIM values
    """
    model.to(DEVICE)
    model.eval()
    
    print("Generating super resolution images...")
    test_sr_images = np.zeros_like(test_hr_images)
    
    with torch.no_grad():
        batch_idx = 0
        for lr_imgs, _ in tqdm(test_loader):
            lr_imgs = lr_imgs.to(DEVICE)
            outputs = model(lr_imgs)
            
            # Convert back to numpy for evaluation
            outputs = outputs.cpu().numpy()
            
            # PyTorch uses [B, C, H, W] format, convert back to [B, H, W, C]
            outputs = outputs.transpose(0, 2, 3, 1)
            
            # Store the super-resolution results
            batch_size = outputs.shape[0]
            start_idx = batch_idx * BATCH_SIZE
            end_idx = min(start_idx + batch_size, len(test_hr_images))
            test_sr_images[start_idx:end_idx] = outputs[:end_idx-start_idx]
            
            batch_idx += 1
    
    # Calculate metrics for each test image
    print("Calculating metrics...")
    individual_psnr = []
    individual_ssim = []
    
    for i in range(len(test_hr_images)):
        try:
            # Convert from [0,1] to [0,255] for metric calculation
            true_img = (test_hr_images[i].squeeze() * 255).astype(np.uint8)
            pred_img = (test_sr_images[i].squeeze() * 255).astype(np.uint8)
            lr_img = (test_lr_images[i].squeeze() * 255).astype(np.uint8)
            
            # Calculate metrics
            psnr_value = psnr(true_img, pred_img, data_range=255)
            ssim_value = ssim(true_img, pred_img, data_range=255)
            
            # Calculate metrics for low-res images as baseline
            psnr_lr = psnr(true_img, lr_img, data_range=255)
            ssim_lr = ssim(true_img, lr_img, data_range=255)
            
            individual_psnr.append(psnr_value)
            individual_ssim.append(ssim_value)
            
            print(f"Image {i+1}: PSNR = {psnr_value:.2f} dB (LR: {psnr_lr:.2f} dB), "
                  f"SSIM = {ssim_value:.4f} (LR: {ssim_lr:.4f})")
        except Exception as e:
            print(f"Error evaluating image {i}: {e}")
    
    # Calculate average metrics
    avg_psnr = np.mean(individual_psnr)
    avg_ssim = np.mean(individual_ssim)
    
    # Calculate metrics for low-res images
    lr_psnr_values = []
    lr_ssim_values = []
    
    for i in range(len(test_hr_images)):
        true_img = (test_hr_images[i].squeeze() * 255).astype(np.uint8)
        lr_img = (test_lr_images[i].squeeze() * 255).astype(np.uint8)
        
        lr_psnr = psnr(true_img, lr_img, data_range=255)
        lr_ssim = ssim(true_img, lr_img, data_range=255)
        
        lr_psnr_values.append(lr_psnr)
        lr_ssim_values.append(lr_ssim)
    
    avg_psnr_lr = np.mean(lr_psnr_values)
    avg_ssim_lr = np.mean(lr_ssim_values)
    
    print(f"Average PSNR: {avg_psnr:.2f} dB (Low-res: {avg_psnr_lr:.2f} dB)")
    print(f"Average SSIM: {avg_ssim:.4f} (Low-res: {avg_ssim_lr:.4f})")
    print(f"PSNR Improvement: {avg_psnr - avg_psnr_lr:.2f} dB")
    print(f"SSIM Improvement: {(avg_ssim - avg_ssim_lr):.4f}")
    
    return test_sr_images, individual_psnr, individual_ssim

## 8. VISUALIZATION FUNCTIONS

In [8]:
def plot_sample_images(lr_images, sr_images, hr_images, sample_count=3):
    """
    Plot sample images: low-res, super-res, and high-res (ground truth)
    """
    plt.figure(figsize=(15, 10))
    for i in range(sample_count):
        plt.subplot(3, sample_count, i + 1)
        plt.imshow(lr_images[i].squeeze(), cmap='gray')
        plt.title('Low Resolution')
        plt.axis('off')
        
        plt.subplot(3, sample_count, i + 1 + sample_count)
        plt.imshow(sr_images[i].squeeze(), cmap='gray')
        plt.title('Super Resolution')
        plt.axis('off')
        
        plt.subplot(3, sample_count, i + 1 + 2*sample_count)
        plt.imshow(hr_images[i].squeeze(), cmap='gray')
        plt.title('High Resolution (Ground Truth)')
        plt.axis('off')
    
    plt.tight_layout()
    plt.savefig('sample_results.png')
    plt.close()


def plot_metrics(psnr_values, ssim_values):
    """
    Plot PSNR and SSIM metrics as bar charts
    """
    plt.figure(figsize=(15, 5))
    plt.bar(range(len(psnr_values)), psnr_values)
    plt.title('PSNR Values for Test Images')
    plt.xlabel('Image Index')
    plt.ylabel('PSNR (dB)')
    plt.savefig('psnr_values.png')
    plt.close()
    
    plt.figure(figsize=(15, 5))
    plt.bar(range(len(ssim_values)), ssim_values)
    plt.title('SSIM Values for Test Images')
    plt.xlabel('Image Index')
    plt.ylabel('SSIM')
    plt.savefig('ssim_values.png')
    plt.close()


def plot_loss_curves(train_losses, val_losses):
    """
    Plot training and validation loss curves
    """
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, len(train_losses) + 1), train_losses, label='Training Loss')
    plt.plot(range(1, len(val_losses) + 1), val_losses, label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig('loss_curves.png')
    plt.close()

## 9. SAVE RESULTS

In [9]:
def save_metrics_to_csv(individual_psnr, individual_ssim):
    """
    Save metrics to CSV file
    """
    metrics_df = pd.DataFrame({
        'Image': range(1, len(individual_psnr) + 1),
        'PSNR': individual_psnr,
        'SSIM': individual_ssim
    })
    metrics_df.to_csv('metrics_results.csv', index=False)
    print("Metrics saved to metrics_results.csv")

## 10. MAIN EXECUTION

In [10]:
def main():
    try:
        print(f"Using device: {DEVICE}")
        
        # 1. Create VDSR model
        model = VDSR(num_layers=NUM_LAYERS, num_filters=NUM_FILTERS)
        print(f"Model created with {NUM_LAYERS} layers and {NUM_FILTERS} filters per layer")
        
        # 2. Define loss function and optimizer
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        
        # Learning rate scheduler
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=LR_DECAY_STEP, gamma=LR_DECAY_GAMMA)
        
        # 3. Prepare training and validation data
        print("Preparing training and validation data...")
        train_loader, val_loader = prepare_data(DATASET_PATH, downscale_factor=DOWNSCALE_FACTOR)
        
        # 4. Train model
        print("Training model...")
        train_losses, val_losses = train_model(
            model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=EPOCHS
        )
        
        # 5. Load best model for evaluation
        checkpoint = torch.load('vdsr_model_best.pth')
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Loaded best model from epoch {checkpoint['epoch']+1} with validation loss {checkpoint['val_loss']:.6f}")
        
        # 6. Load test data
        print("Loading test data...")
        test_loader, test_hr_images, test_lr_images = load_test_data(DATASET_PATH, downscale_factor=DOWNSCALE_FACTOR)
        
        # 7. Evaluate model
        test_sr_images, individual_psnr, individual_ssim = evaluate_model(
            model, test_loader, test_hr_images, test_lr_images
        )
        
        # 8. Plot results
        plot_sample_images(test_lr_images[:10], test_sr_images[:10], test_hr_images[:10])
        plot_metrics(individual_psnr, individual_ssim)
        plot_loss_curves(train_losses, val_losses)
        
        # 9. Save metrics to CSV
        save_metrics_to_csv(individual_psnr, individual_ssim)
        
        print("Done! All results saved to disk.")
        
    except Exception as e:
        print(f"An error occurred in the main execution: {e}")

## 11. ENTRY POINT

In [11]:
if __name__ == "__main__":
    main()

Using device: cuda
Model created with 20 layers and 64 filters per layer
Preparing training and validation data...
Found 0 NORMAL images


0it [00:00, ?it/s]


Found 0 PNEUMONIA images


0it [00:00, ?it/s]


Total HR images: (0,)


0it [00:00, ?it/s]

An error occurred in the main execution: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.
